#### Data ingestion Layer:

 * get documents
 * create chunks
* embeddings
* call the llm transformer instance to generate our
knowledge graph(KG)

#### Load Libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
import ollama
import re # regex - regular experessions 
from langchain_ollama import ChatOllama
from langchain_neo4j import Neo4jGraph, LLMGraphTransformer

C:\Users\HP\AppData\Local\Temp\ipykernel_7764\3250296904.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader = PyPDFLoader("data/Darwinism.pdf")
pages = loader.load()

In [3]:
pages

[Document(metadata={'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-16T07:09:35+00:00', 'title': 'Darwinism - Wikipedia', 'moddate': '2026-10-16T07:09:35+00:00', 'source': 'data/Darwinism.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Charles Darwin in 1868\nDarwinism\nDarwinism is a term used to describe a theory of biological\nevolution developed by the English naturalist Charles Darwin\n(1809–1882) and his contemporaries. The theory states that\nall species of organisms arise and develop through the natural\nselection of small, inherited variations that increase the\nindividual\'s ability to compete, survive, and reproduce. Also\ncalled Darwinian theory, it originally included the broad\nconcepts of transmutation of species or of evolution which\ngained general scientific acceptance after Darwin published\nOn the Origin of Species in

In [4]:
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)     # re-join words hyphenated across lines
    text = text.replace('•', '\n- ')           # bullets as their own lines
    text = re.sub(r'[ \t]+', ' ', text)        # collapse spaces/tabs only
    text = re.sub(r'\n{3,}', '\n\n', text)     # cap blank lines
    return text.strip()

#### Split and create chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap= 80) # created an instance of the text splitter
chunks = splitter.split_documents(pages)

In [6]:
# embedding function
def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings[0]

In [7]:
doc_embeddings = [] 

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 


In [8]:
with open("doc_embeddings.txt", "w") as f:
    for item in doc_embeddings:
        f.write(f"{item}\n")

In [9]:
# creates an instance of the model we will use low temperature and high content window
gemma_llm = ChatOllama(model="gemma3:latest", temperature=0)
mistral_llm =  ChatOllama(model="mistral:7b", temperature=0)


In [10]:
# connect to neo4J DB
graph = Neo4jGraph(url="neo4j://127.0.0.1:7687", username="neo4j", password="neo4j12345678")

gemma_transformer = LLMGraphTransformer(llm=gemma_llm)
mistral_transformer = LLMGraphTransformer(llm=mistral_llm)

#### Create knowledge  graph

In [11]:
len(chunks)

38

In [ ]:
# graph_documents = transformer.convert_to_graph_documents([chunks[0]]) 
graph_documents = []
for chunk in chunks[:4]:  
    graph_documents.append(gemma_transformer.convert_to_graph_documents([chunk]))
# TODO: Try out direct transformationa nd laod i.e graph_documents.append(deep_seek_transformer.convert_to_graph_documents([chunks[:]]))

In [13]:
graph_documents = gemma_transformer.convert_to_graph_documents(chunks)

In [14]:
graph_documents

[GraphDocument(nodes=[Node(id='Charles Darwin', type='Person', properties={}), Node(id='Darwinism', type='Theory', properties={}), Node(id='1868', type='Date', properties={}), Node(id='1809', type='Date', properties={}), Node(id='1882', type='Date', properties={}), Node(id='Thomas Henry Huxley', type='Person', properties={})], relationships=[Relationship(source=Node(id='Darwinism', type='Theory', properties={}), target=Node(id='Charles Darwin', type='Person', properties={}), type='DEVELOPED_BY', properties={}), Relationship(source=Node(id='Charles Darwin', type='Person', properties={}), target=Node(id='1809', type='Date', properties={}), type='BORN_IN', properties={}), Relationship(source=Node(id='Charles Darwin', type='Person', properties={}), target=Node(id='1882', type='Date', properties={}), type='DIED_IN', properties={}), Relationship(source=Node(id='Darwinism', type='Theory', properties={}), target=Node(id='1868', type='Date', properties={}), type='FIRST_PUBLISHED', properties={}

In [15]:
with open("graph_doc.txt", "w") as f:
    for item in graph_documents:
        f.write(f"{item}\n")

In [21]:
for graph_doc in graph_documents:
    graph.add_graph_documents([graph_doc], baseEntityLabel=True, include_source=True)

In [22]:
def kg_answer(question, graph, model="llama3.2:1b"):
    # Step 1: ask the LLM which single entity the question is about
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": "What is the single main entity (person, place, or concept) this question is about? Reply with just the name, nothing else."},
            {"role": "user", "content": question},
        ],
        options={"temperature": 0},
    )
    entity = response["message"]["content"].strip()

    # Step 2: query the graph for anything connected to it
    facts = graph.query(
        """
        MATCH (n)-[r]-(m)
        WHERE toLower(n.id) CONTAINS toLower($name)
        RETURN n.id AS entity, type(r) AS relation, m.id AS related
        LIMIT 20
        """,
        {"name": entity}
    )

    if not facts:
        return f"I couldn't find '{entity}' in the knowledge graph."

    context = "\n".join(f"{f['entity']} -{f['relation']}-> {f['related']}" for f in facts)

    # Step 3: generate the answer from those facts
    prompt = f"""Answer the question using only the facts below.
If the facts don't answer it, say you don't know.

Facts:
{context}

Question: {question}"""

    final = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    return final["message"]["content"]

In [23]:
print(kg_answer("What is Darwinian theory?", graph=graph, model="gemma3:latest"))

Based on the provided facts, it appears that Darwinian theory is discussed in the book “Darwin And After Darwin: An Exposition Of The Darwinian Theory And A Discussion Of Post-Darwinian Questions” by John George Romanes. 

Additionally, a "Darwinian Theory of Beauty" is discussed in a book by Denis Dutton. 

I don't know the specific details of what the theory is, only that it is being discussed in these two works.


In [24]:
print(kg_answer("Who is Huxley?", graph=graph, model="gemma3:latest"))

Based on the facts provided, Thomas Henry Huxley was:

*   The author of *Darwiniana: Essays*.
*   He coined the term *Darwinism*.
*   He was associated with *Darwinism* and *Darwin*.
*   He supported *Lamarckism*.
*   He reviewed *On The Origin Of Species* in April 1860.
*   He published in *The Westminster Review*.
*   He contributed to *Darwin On The Origin Of Species*.
*   He responded to *Darwin'S Theory*.


In [ ]:

print(kg_answer("Who is Huxley?", graph=graph, model="mistral:7b"))

 Thomas Henry Huxley, often referred to as 'Darwin's Bulldog', was an English biologist, philosopher, and physician. He is best known for his support of Charles Darwin's theory of evolution and the coining of the term "Darwinism." He also wrote and contributed to various works, including "Darwiniana: Essays," a review of Darwin's "On The Origin Of Species," and the "Westminster Review." He was associated with Darwin and, contrary to his support for Darwinism, he initially supported Lamarckism.


: 